##### Import statements:

In [7]:
import numpy as np
import matplotlib.pyplot
import torch
import torch.nn as nn
import torch.nn.functional as F 
from torch.autograd import Variable
from torch.utils.data import DataLoader
import torch.multiprocessing as mp
import time

##### Define parameters:

In [8]:
# Data parameters:
n_feat = 80
n_obs = 320
n_classes = 2
data_noise = 0.1

# Model parameters:
n_inp = n_feat
n_hidden = 100
sigma_init = 1
sigma_noise = 0.1
batch_size = 64
n_epochs = 500
lr = 0.001
beta_ces = 10**np.arange(0, 5, 0.5)
beta_sp = 1
p_norm = 2

# Compute paramters:
gpu = True

##### Define model class:

In [19]:
class Mdl(nn.Module):
    def __init__(self,n_inp,n_hidden, n_classes, sigma_init):    
        super(Mdl,self).__init__()
        self.n_inp=n_inp
        self.n_hidden=n_hidden
        self.n_classes=n_classes
        self.sigma_init=sigma_init
        self.enc=torch.nn.Linear(n_inp,n_hidden)
        self.dec=torch.nn.Linear(n_hidden,n_classes)
        self.apply(self._init_weights)
        
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            module.weight.data.normal_(mean=0.0, std=self.sigma_init)
            if module.bias is not None:
                module.bias.data.normal_(mean=0.0, std=self.sigma_init)

    def forward(self,x,sigma_noise,gpu=False):
        if not gpu:
            x_hidden = F.relu(self.enc(x))+sigma_noise*torch.randn(x.size(0),self.n_hidden)
        else:
            x_hidden = F.relu(self.enc(x))+sigma_noise*torch.randn(x.size(0),self.n_hidden).to('cuda')
        x = self.dec(x_hidden)
        return x,x_hidden


def train_model(mdl, X_torch, labels_torch, n_epochs=100, beta_ce=1, beta_sp=1, sigma_init=1, sigma_noise=0.1, batch_size=64, gpu=False):

    # Get dimensions:
    #n_inp = X.shape[1]
    #n_classes = labels.shape[1]

    # Initialize model, optimizer:
    #mdl = Mdl(n_inp, n_hidden, n_classes, sigma_init)
    loss_ce = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(mdl.parameters(), lr=lr)

    # Convert data to torch:
    #X_torch = Variable(torch.from_numpy(X))
    #labels_torch = Variable(torch.from_numpy(labels.astype(np.float32)))

    # Move stuff to GPU if necessary:
    #if gpu:
    #    mdl.to('cuda')
    #    X_torch = X_torch.to('cuda')
    #    labels_torch = labels_torch.to('cuda')

    # Initialize loader:
    dset = torch.utils.data.TensorDataset(X_torch, labels_torch)
    data_loader = DataLoader(dset, batch_size=batch_size, shuffle=True)
    
    # Iterate over training epochs:
    for t in np.arange(n_epochs):

        # Iterate over training batches:
        for batch_idx, (curr_X, curr_labels) in enumerate(data_loader):
    
            print('Training epoch {} of {}, batch {} of {}...'.format(t+1, n_epochs, batch_idx+1, len(data_loader)))
            
            # Reset optimizer:
            optimizer.zero_grad()
    
            # Forward pass:
            #curr_X.to('cuda')
            output = mdl(curr_X, sigma_noise, gpu=gpu)
    
            # Compute loss:
            L_ce = loss_ce(output[0], np.squeeze(curr_labels))
            L_sp = sparsity_loss(output[1], p_norm)
            L = beta_ce*L_ce + beta_sp*L_sp 
    
            # Backprop:
            L.backward()
            optimizer.step()


def sparsity_loss(data,p):
    loss=torch.mean(torch.pow(abs(data),p),axis=(0,1))
    return loss

##### Generate data:

In [20]:
# Define centroid for each class:
mus = []
for i in np.arange(n_classes):
    curr_mu = np.random.randn(n_feat).astype(np.float32)
    mus.append(curr_mu)

# Define covariance matrix:
C = np.random.randn(n_feat, n_feat).astype(np.float32)

# Generate labels:
labels = np.random.choice(np.arange(n_classes), n_obs)
labels = np.expand_dims(labels, axis=1)

# Generate data:
X = np.array([mus[x] for x in np.squeeze(labels)])

# Add noise:
Eps = np.random.randn(n_obs, n_feat).astype(np.float32)
Eps = np.matmul(C, Eps.T).T 
X = X + Eps

# Convert data to torch:
X_torch = Variable(torch.from_numpy(X))
labels_torch = Variable(torch.from_numpy(labels)).long()

# Move to gpu if necessary:
if gpu:
    X_torch = X_torch.to('cuda')
    labels_torch = labels_torch.to('cuda')

##### Iterate over training epochs:

In [ ]:
start_train = time.time()
processes = []
for b, beta_ce in enumerate(beta_ces):

    print('Running hyperparameter value {} out of {}...'.format(b+1, len(beta_ces)))
    mdl = Mdl(n_inp, n_hidden, n_classes, sigma_init)
    if gpu:
        mdl.to('cuda')
    p = mp.Process(target=train_model, args=(mdl, X_torch, labels_torch, n_epochs, beta_ce, beta_sp, sigma_init, sigma_noise, batch_size, gpu))
    #train_model(X, labels, n_hidden=n_hidden, n_epochs=n_epochs, beta_ce=beta_ce, beta_sp=beta_sp, sigma_init=sigma_init, 
    #            sigma_noise=sigma_noise, batch_size=batch_size, gpu=gpu)
    p.start()
    processes.append(p)

for p in processes:
    p.join()

stop_train = time.time()
print('Training duration : {} s'.format(stop_train - start_train))

Running hyperparameter value 1 out of 10...
Running hyperparameter value 2 out of 10...
Running hyperparameter value 3 out of 10...
Running hyperparameter value 4 out of 10...
Running hyperparameter value 5 out of 10...
Running hyperparameter value 6 out of 10...
Running hyperparameter value 7 out of 10...
Running hyperparameter value 8 out of 10...
Running hyperparameter value 9 out of 10...
Running hyperparameter value 10 out of 10...
Training epoch 1 of 500, batch 1 of 5...Training epoch 1 of 500, batch 1 of 5...Training epoch 1 of 500, batch 1 of 5...

Training epoch 1 of 500, batch 1 of 5...Training epoch 1 of 500, batch 1 of 5...
Training epoch 1 of 500, batch 1 of 5...
Training epoch 1 of 500, batch 1 of 5...
Training epoch 1 of 500, batch 1 of 5...Training epoch 1 of 500, batch 1 of 5...Training epoch 1 of 500, batch 1 of 5...




Training epoch 1 of 500, batch 2 of 5...
Training epoch 1 of 500, batch 2 of 5...
Training epoch 1 of 500, batch 3 of 5...
Training epoch 1 of 500, b

In [ ]:
print('Done')